<a href="https://colab.research.google.com/github/hvn2k/Twitter-Analysis/blob/main/Copy_of_Twtr_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Program to get sentiment of Bitcoin from Twitter users**

Importing the libraries

In [ ]:
import tweepy
from textblob import TextBlob
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

Twitter API credentials

In [ ]:
consumer_key = 'YOUR-API-KEY'
consumer_secret = 'YOUR-API-SECRET-KEY'
access_token = 'YOUR-ACCESS-TOKEN'
access_secret = 'YOUR-ACCESS-TOKEN-SECRET'

Authenticator

In [ ]:
auth = tweepy.OAuthHandler(consumer_key,consumer_secret)
auth.set_access_token(access_token,access_secret)
api = tweepy.API(auth, wait_on_rate_limit=True)

Gather Tweets

In [ ]:
search_term = '#bitcoin -filter:retweets'
tweets = tweepy.Cursor(api.search, q=search_term, lang='en', since_id='TWEET-ID', tweet_mode= 'extended').items(500) #You can change the number of items accordingly. Here items represents the number of tweets
all_tweets = [tweet.full_text for tweet in tweets]

Create a dataframe to store the tweets

In [ ]:
df = pd.DataFrame(all_tweets, columns=['Tweets'])
df.head(5)

Function for cleaning the tweets

In [ ]:
def cleanTwt(twt):
  twt = re.sub('#bitcoin', 'bitcoin', twt)
  twt = re.sub('#Bitcoin', 'Bitcoin', twt)
  twt = re.sub('#[A-Za-z0-9]+', '', twt)
  twt = re.sub('\\n', '', twt)
  twt = re.sub('https?:\/\/\S+', '', twt)
  return twt

Cleaning the tweets

In [ ]:
df['Cleaned_Tweets'] = df['Tweets'].apply(cleanTwt)
df.head(5)

Creating a function to get the subjectivity

In [ ]:
def getSubjectivity(twt):
  return TextBlob(twt).sentiment.subjectivity

def getPolarity(twt):
  return TextBlob(twt).sentiment.polarity

df['Subjectivity'] = df['Cleaned_Tweets'].apply(getSubjectivity)
df['Polarity'] = df['Cleaned_Tweets'].apply(getPolarity)

df.head()

Creating function to get the text sentiment

In [ ]:
def getSentiment(score):
  if score < 0:
    return 'Negative'
  elif score == 0:
    return 'Neutral'
  else:
    return 'Positive'

Creating a column to store the text sentiment

In [ ]:
df['Sentiment'] = df['Polarity'].apply(getSentiment)

df.head()

Plot to show subjectivity and polarity

In [ ]:
plt.figure(figsize=(8,6))
for i in range(0, df.shape[0]):
  plt.scatter(df['Polarity'][i], df['Subjectivity'][i], color='Purple')
plt.title('Sentiment Analysis Scatter Plot for #bitcoin')
plt.xlabel('Polarity')
plt.ylabel('Subjectivity')
plt.show()

Creating a bar chart to show the count of Positive, Negative and Neutral sentiments

In [ ]:
df['Sentiment'].value_counts().plot(kind='bar')
plt.title('Sentiment Analysis Bar Plot for #bitcoin')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()